# L01 · MDP + Policy Gradient + REINFORCE

对应 lecture: `lectures/01-mdp-policy.md`

**目标**：
- 跑通 REINFORCE on CartPole（手写版本）
- 观察单 episode reward 噪声
- 对比 vanilla vs with-baseline 两个变体


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import torch
import matplotlib.pyplot as plt
import gymnasium as gym

from common import set_seed, compute_returns
from reinforce_minimal import PolicyNet, run_episode, compute_loss

## 1. 跑 200 ep REINFORCE（vanilla, 无 baseline）

In [ ]:
def run_reinforce(use_baseline: bool, n_episodes: int = 200, seed: int = 0):
    set_seed(seed)
    env = gym.make('CartPole-v1')
    env.reset(seed=seed)
    policy = PolicyNet(state_dim=4, hidden=64, n_actions=2)
    opt = torch.optim.Adam(policy.parameters(), lr=1e-3)

    returns = []
    for ep in range(n_episodes):
        log_probs, rewards, ret = run_episode(env, policy, device='cpu')
        loss = compute_loss(log_probs, rewards, gamma=0.99, use_baseline=use_baseline)
        opt.zero_grad(); loss.backward(); opt.step()
        returns.append(ret)
    env.close()
    return returns

R_van = run_reinforce(use_baseline=False)
R_bsl = run_reinforce(use_baseline=True)

## 2. 画 reward 曲线对比（含 moving avg）

In [ ]:
def moving_avg(x, w=10):
    return np.convolve(x, np.ones(w)/w, mode='valid')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(R_van, alpha=0.3, label='raw')
ax[0].plot(np.arange(9, len(R_van)), moving_avg(R_van), label='moving avg 10')
ax[0].set_title('REINFORCE vanilla')
ax[0].set_xlabel('episode'); ax[0].set_ylabel('return'); ax[0].legend()

ax[1].plot(R_bsl, alpha=0.3, label='raw')
ax[1].plot(np.arange(9, len(R_bsl)), moving_avg(R_bsl), label='moving avg 10')
ax[1].set_title('REINFORCE + episode-mean baseline')
ax[1].set_xlabel('episode'); ax[1].set_ylabel('return'); ax[1].legend()

plt.tight_layout(); plt.show()
print(f'vanilla last-50 mean: {np.mean(R_van[-50:]):.1f}')
print(f'baseline last-50 mean: {np.mean(R_bsl[-50:]):.1f}')

## 3. 单 episode 内 G_t 与 ∇ log π 的方差观察

在最后一条 episode 上取 `G_t`，观察其抖动。


In [ ]:
set_seed(0)
env = gym.make('CartPole-v1')
env.reset(seed=0)
policy = PolicyNet(state_dim=4, hidden=64, n_actions=2)

log_probs, rewards, ret = run_episode(env, policy)
dones = [False]*(len(rewards)-1)+[True]
G = compute_returns(rewards, dones, gamma=0.99)

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(G); ax[0].set_title(f'G_t (this episode, return={ret:.0f})'); ax[0].set_xlabel('t')
ax[1].hist(G, bins=20); ax[1].set_title('histogram of G_t'); ax[1].set_xlabel('G_t')
plt.tight_layout(); plt.show()

## 4. 自我检查

- [ ] vanilla 与 baseline 两条曲线最后 50 ep 均值差 ≥ 20
- [ ] G_t 的方差肉眼可见（episode 长度 > 100 时 G_t 跨度数十）
- [ ] 理解 PG 公式 `g = ∇log π · G_t`
